# I. Observarea statistică

## Studiul înregistrărilor de vânzări din trei cafenele din New York

Analiza folosește fișierul local `Coffee Shop Sales.csv` și se referă exclusiv la înregistrările din cele trei locații, între **1 ianuarie și 30 iunie 2023**. Proveniența inițială și caracterul exhaustiv al bazei față de vânzările efective nu sunt documentate; rezultatele descriu fișierul disponibil.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
date = pd.read_csv('Coffee Shop Sales.csv')
coloane_sursa = date.columns.tolist()
date['transaction_date'] = pd.to_datetime(
    date['transaction_date'], format='%m/%d/%y', errors='raise'
)
date.head()

### 1.1. Scopul observării statistice

Scopul observării statistice este analiza structurii și dinamicii tranzacțiilor de vânzare efectuate în cele trei cafenele din New York, în perioada 1 ianuarie 2023 – 30 iunie 2023. Observarea urmărește volumul tranzacțiilor, cantitățile vândute, prețurile unitare, structura produselor și distribuția vânzărilor pe locații și în timp.

### 1.2. Populația statistică

Populația accesibilă cuprinde toate înregistrările din fișier pentru locațiile Astoria, Hell's Kitchen și Lower Manhattan. Unitatea de analiză este **înregistrarea cu un `transaction_id` distinct**. Identificatorii sunt unici în acest fișier, însă documentația disponibilă nu confirmă dacă fiecare cod reprezintă un bon complet ori o poziție de bon. Prin urmare, indicatorii de mai jos sunt raportați la înregistrări, fără a afirma că surprind toate vânzările efective ale celor trei cafenele.

### 1.3. Metoda de observare

S-a utilizat observarea indirectă, documentară, a unei baze de date secundare. Sunt analizate toate rândurile disponibile, fără eșantionare sau eliminări. Nu există informații care să permită evaluarea erorilor de acoperire ori a modului inițial de colectare.

In [ ]:
zile_asteptate = pd.date_range(
    date['transaction_date'].min(), date['transaction_date'].max(), freq='D'
)
zile_lipsa = zile_asteptate.difference(date['transaction_date'].unique())
verificare_esantion = pd.Series({
    "Număr de înregistrări": len(date),
    "Identificatori unici": date['transaction_id'].nunique(),
    "Rânduri duplicate integral": int(date.duplicated().sum()),
    "Identificatori repetați": int(date['transaction_id'].duplicated().sum()),
    "Prima zi observată": date['transaction_date'].min().date(),
    "Ultima zi observată": date['transaction_date'].max().date(),
    "Număr de zile observate": date['transaction_date'].nunique(),
    "Zile lipsă în interval": len(zile_lipsa),
    "Număr de locații": date['store_location'].nunique(),
    "Valori lipsă": int(date.isna().sum().sum()),
    "Cantități sau prețuri nepozitive": int((
        (date['transaction_qty'] <= 0) | (date['unit_price'] <= 0)
    ).sum())
})
assert date['transaction_id'].is_unique
assert not date.isna().any().any()
assert len(zile_lipsa) == 0
assert (date['transaction_qty'] > 0).all() and (date['unit_price'] > 0).all()
assert date.groupby('store_id')['store_location'].nunique().eq(1).all()
display(verificare_esantion)
display(date.isna().sum().rename('Valori lipsă pe variabilă').to_frame())

### 1.4. Eșantionul observat

Fișierul conține **149.116 înregistrări cu identificator unic** pe parcursul a 181 de zile consecutive în cele trei locații. Termenul „eșantion observat” desemnează aici datele accesibile pentru studiu, nu un eșantion probabilistic. Calculele sunt descriptive pentru fișier; nu se generalizează automat la toate cafenelele din New York sau la alte perioade. Nu sunt calculate intervale de încredere ori teste de semnificație în lipsa unui plan de eșantionare sau a unui model probabilistic justificat.

### 1.5. Caracteristicile statistice

| Grup | Caracteristici / variabile | Rol în analiză |
|---|---|---|
| Identificatori tehnici | `transaction_id`, `store_id`, `product_id` | Identifică tranzacția, locația și produsul |
| Temporale | `transaction_date`, `transaction_time` | Data și ora tranzacției |
| Cantitative | `transaction_qty`, `unit_price` | Cantitatea cumpărată și prețul unitar |
| Calitative | `store_location`, `product_category`, `product_type`, `product_detail` | Locația și clasificarea produsului |

Identificatorii tehnici permit evidența observațiilor, dar nu sunt utilizați ca variabile analitice.

### 1.6. Scalele de măsurare

| Caracteristica statistică | Tip | Scala de măsurare |
|---|---|---|
| `store_location`, `product_category`, `product_type`, `product_detail` | Calitativă | Nominală |
| `transaction_date`, `transaction_time` | Temporală | De interval pentru diferențe temporale |
| `transaction_qty` | Cantitativă discretă | De raport |
| `unit_price` | Cantitativă numerică, în trepte monetare | De raport |

`transaction_id`, `store_id` și `product_id` sunt coduri nominale, nu măsuri cantitative. Setul nu conține o caracteristică ordinală explicită. Venitul calculat mai jos este o variabilă derivată, nu o coloană a sursei.

In [ ]:
locatii = (
    date[["store_id", "store_location"]]
    .drop_duplicates()
    .sort_values("store_id")
)

structura_date = pd.Series({
    "Număr de variabile": date.shape[1],
    "Număr de produse": date["product_id"].nunique(),
    "Număr de categorii": date["product_category"].nunique(),
    "Valori lipsă": int(date.isna().sum().sum())
})

display(locatii)
display(structura_date)

### 1.7. Sursa de date și limitele verificării

Sursa efectiv utilizată este fișierul local `Coffee Shop Sales.csv`, care conține 11 variabile. Nu este documentată în materialele disponibile o adresă exactă de descărcare, autorul bazei, licența ori metoda colectării inițiale. O referire generică la Kaggle nu poate verifica proveniența. Integritatea internă a fișierului este testată mai sus, dar aceasta nu dovedește autenticitatea economică sau acoperirea completă a tranzacțiilor reale.

# II. Sistematizarea şi prezentarea datelor observate

### 2.1. Tabelul descriptiv

Sunt prezentate primele zece înregistrări din fișier, dimensiunea sursei, tipurile celor 11 variabile și valorile lipsă. Tabelul complet rămâne în `Coffee Shop Sales.csv`; nu sunt aplicate filtre sau imputări.

In [ ]:
rezumat_date = pd.DataFrame({
    "Indicator": ["Număr de observații", "Număr de variabile", "Valori lipsă"],
    "Valoare": [len(date), date.shape[1], int(date.isna().sum().sum())]
})

display(date.head(10))
display(rezumat_date)
display(date.dtypes.rename("Tip de date").to_frame())

### 2.2. Clasificarea unităților statistice observate și calcularea ponderii în grupe

Frecvențele pentru locație, categorie și tip de produs numără **înregistrări**, iar ponderea lor are ca numitor 149.116. Cantitățile și încasările sunt măsuri diferite. Pentru fiecare rând se calculează `venit_usd = transaction_qty × unit_price`; aceasta este valoarea rezultată din câmpurile fișierului, fără taxe, reduceri sau retururi documentate. Nu se presupune că reprezintă încasări contabile auditate.

In [ ]:
def tabel_frecvente(serie, denumire):
    rezultat = (
        serie.value_counts(dropna=False)
        .rename_axis(denumire)
        .reset_index(name="Frecvență")
    )
    rezultat["Pondere (%)"] = rezultat["Frecvență"] / len(serie) * 100
    total = pd.DataFrame({
        denumire: ["Total"], "Frecvență": [rezultat["Frecvență"].sum()],
        "Pondere (%)": [100.0]
    })
    return pd.concat([rezultat, total], ignore_index=True)

tabel_locatii = tabel_frecvente(date['store_location'], 'Locație')
tabel_categorii = tabel_frecvente(date['product_category'], 'Categorie produs')
tabel_tipuri = tabel_frecvente(date['product_type'], 'Tip produs')

date_analiza = date.assign(
    venit_usd=date['transaction_qty'] * date['unit_price']
)
assert len(date_analiza) == len(date)
assert date_analiza['transaction_id'].is_unique
assert date_analiza['transaction_date'].equals(date['transaction_date'])
assert not date_analiza.isna().any().any()

def tabel_masuri(coloana):
    rezultat = date_analiza.groupby(coloana).agg(
        Inregistrari=('transaction_id', 'size'),
        Unitati=('transaction_qty', 'sum'),
        Venit_USD=('venit_usd', 'sum')
    ).sort_values('Inregistrari', ascending=False)
    for masura in ['Inregistrari', 'Unitati', 'Venit_USD']:
        rezultat[f'Pondere {masura} (%)'] = (
            rezultat[masura] / rezultat[masura].sum() * 100
        )
    return rezultat

masuri_locatii = tabel_masuri('store_location')
masuri_categorii = tabel_masuri('product_category')
assert masuri_locatii['Inregistrari'].sum() == len(date)
assert masuri_locatii['Unitati'].sum() == date['transaction_qty'].sum()
assert np.isclose(masuri_locatii['Venit_USD'].sum(), date_analiza['venit_usd'].sum())
display(tabel_locatii)
display(tabel_categorii)
display(tabel_tipuri)
display(masuri_locatii.round(2))
display(masuri_categorii.round(2))

In [ ]:
def diagrama_frecvente(tabel, coloana, titlu, top_n=None):
    valori = tabel.loc[
        tabel[coloana] != "Total",
        [coloana, "Pondere (%)"]
    ].sort_values("Pondere (%)", ascending=False).copy()

    if top_n is not None and len(valori) > top_n:
        pondere_alte = valori.iloc[top_n:]["Pondere (%)"].sum()
        valori = pd.concat([
            valori.iloc[:top_n],
            pd.DataFrame({
                coloana: ["Alte tipuri"],
                "Pondere (%)": [pondere_alte]
            })
        ], ignore_index=True)

    valori = valori.iloc[::-1]

    fig, ax = plt.subplots(figsize=(10, max(3, len(valori) * 0.5)))
    bare = ax.barh(valori[coloana].astype(str), valori["Pondere (%)"])
    ax.bar_label(bare, fmt="%.1f%%", padding=3)
    ax.set_xlim(0, valori["Pondere (%)"].max() * 1.2)
    ax.set_xlabel("Pondere (%)")
    ax.set_title(titlu)
    plt.tight_layout()
    plt.show()

diagrama_frecvente(tabel_locatii, "Locație", "Distribuția pe locații")
diagrama_frecvente(tabel_categorii, "Categorie produs", "Distribuția pe categorii de produse")
diagrama_frecvente(tabel_tipuri, "Tip produs", "Cele mai frecvente tipuri de produse", top_n=10)

### 2.3. Gruparea unităților statistice după caracteristici cantitative

Cantitatea `transaction_qty` se grupează pe valorile întregi de la minim la maxim; valorile neobservate rămân vizibile cu frecvență zero. Pentru `unit_price` se folosește regula Sturges, `k = ceil(1 + 3,322 × lg(n))` și lățimea `h = (x_max − x_min) / k`. Intervalele au limita dreaptă inclusă, cu excepția primului, care include și minimul. Etichetele folosesc limitele calculate, nu marginile rotunjite intern de `pd.cut`. Prețurile reprezintă amestecul unor categorii de produse diferite; gruparea nu descrie separat prețul unei băuturi obișnuite.

In [ ]:
cantitati = pd.Index(range(
    date['transaction_qty'].min(), date['transaction_qty'].max() + 1
))
grupare_cantitate = (
    date['transaction_qty'].value_counts().reindex(cantitati, fill_value=0)
    .rename_axis('Cantitate cumpărată').reset_index(name='Frecvență')
)
grupare_cantitate['Pondere (%)'] = grupare_cantitate['Frecvență'] / len(date) * 100
grupare_cantitate = pd.concat([
    grupare_cantitate,
    pd.DataFrame({'Cantitate cumpărată': ['Total'],
                  'Frecvență': [len(date)], 'Pondere (%)': [100.0]})
], ignore_index=True)

n = len(date)
x_min, x_max = date['unit_price'].min(), date['unit_price'].max()
amplitudine = x_max - x_min
k = int(np.ceil(1 + 3.322 * np.log10(n)))
h = amplitudine / k
limite = np.linspace(x_min, x_max, k + 1)
intervale_pret = pd.cut(date['unit_price'], bins=limite, include_lowest=True)
etichete_pret = [
    f"{'[' if i == 0 else '('}{limite[i]:.4f}, {limite[i+1]:.4f}]"
    for i in range(k)
]
grupare_pret = pd.DataFrame({
    'Interval preț unitar (USD)': etichete_pret,
    'Frecvență': intervale_pret.value_counts(sort=False).to_numpy()
})
grupare_pret['Pondere (%)'] = grupare_pret['Frecvență'] / n * 100
grupare_pret = pd.concat([
    grupare_pret,
    pd.DataFrame({'Interval preț unitar (USD)': ['Total'],
                  'Frecvență': [n], 'Pondere (%)': [100.0]})
], ignore_index=True)
parametri_grupare = pd.Series({
    'x_min (USD)': x_min, 'x_max (USD)': x_max,
    'Amplitudinea A': amplitudine,
    'Număr de grupe k (Sturges)': k, 'Lățimea intervalului h': h,
    'Pondere în primele două grupe (%)': grupare_pret.iloc[:2]['Pondere (%)'].sum()
})
assert intervale_pret.notna().all()
assert grupare_cantitate.iloc[:-1]['Frecvență'].sum() == n
assert grupare_pret.iloc[:-1]['Frecvență'].sum() == n
display(parametri_grupare)
display(grupare_cantitate)
display(grupare_pret)

### 2.4. Prezentarea grafică a distribuțiilor și a structurii

Graficele folosesc aceleași frecvențe ca tabelele. În diagrama circulară, cantitățile rare de cel puțin patru unități sunt reunite doar pentru lizibilitate; tabelul păstrează fiecare valoare. Intervalele Sturges sunt păstrate integral, iar un grafic suplimentar mărește zona prețurilor de până la 5,50 USD. Acest prag este folosit numai în graficul de detaliu, nu exclude rânduri din calcule.

In [ ]:
date_cantitate = grupare_cantitate.iloc[:-1].copy()
fig, axe = plt.subplots(1, 2, figsize=(13, 4.5))
axe[0].bar(date_cantitate['Cantitate cumpărată'].astype(str),
           date_cantitate['Frecvență'])
axe[0].set(title='Înregistrări după cantitate', xlabel='Unități pe înregistrare',
           ylabel='Număr de înregistrări')
frecvente_pie = date_cantitate.loc[
    date_cantitate['Cantitate cumpărată'].isin([1, 2, 3]),
    ['Cantitate cumpărată', 'Frecvență']
].copy()
frecvente_pie.loc[len(frecvente_pie)] = ['4 sau mai mult',
    date_cantitate.loc[date_cantitate['Cantitate cumpărată'] >= 4, 'Frecvență'].sum()]
axe[1].pie(frecvente_pie['Frecvență'],
           labels=frecvente_pie['Cantitate cumpărată'].astype(str),
           autopct=lambda p: f'{p:.2f}%' if p >= 0.05 else '', startangle=90)
axe[1].set_title('Structura înregistrărilor după cantitate')
plt.tight_layout()
plt.show()

In [ ]:
date_pret = grupare_pret.iloc[:-1].copy()
fig, axe = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
axe[0].bar(date_pret['Interval preț unitar (USD)'], date_pret['Frecvență'])
axe[0].set(title='Înregistrări după intervalul prețului unitar',
           ylabel='Număr de înregistrări')
axe[1].bar(date_pret['Interval preț unitar (USD)'], date_pret['Pondere (%)'])
axe[1].set(title='Structura după intervalul prețului unitar',
           xlabel='Interval preț unitar (USD)', ylabel='Pondere (%)')
for ax in axe:
    ax.tick_params(axis='x', rotation=70)
plt.tight_layout()
plt.show()

prag_detaliu = 5.50
preturi_detaliu = date.loc[date['unit_price'] <= prag_detaliu, 'unit_price']
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(preturi_detaliu, bins=np.arange(0.75, 5.76, 0.25), edgecolor='white')
ax.set(title=f'Prețuri unitare până la {prag_detaliu:.2f} USD '
             f'({len(preturi_detaliu) / n:.1%} din înregistrări)',
       xlabel='Preț unitar (USD)', ylabel='Număr de înregistrări')
plt.tight_layout()
plt.show()

# III. Analiza statistică a tendinței centrale și a variației

Indicatorii descriu populația **din fișier**, fără extrapolare probabilistică. Pentru cantitate și preț sunt raportate atât valorile exacte calculate din observații, cât și, unde este relevant, aproximațiile bazate pe grupările din capitolul II. Prețul mediu pe înregistrare nu este prețul mediu pe unitate vândută și nu este valoarea medie a unei încasări.

### 3.1. Indicatorii medii pentru fiecare distribuție

Media aritmetică exactă este calculată din fiecare rând. Pentru cantitate, media ponderată cu frecvențele valorilor întregi coincide cu media exactă. Pentru preț, media pe intervale folosește centrele claselor și este doar o aproximație, deoarece prețurile dintr-un interval nu sunt distribuite uniform. Se mai arată media geometrică și cea armonică, posibile fiindcă toate valorile sunt pozitive; pentru interpretarea uzuală se folosește media aritmetică.

In [ ]:
def indicatori_medii(serie):
    valori = serie.to_numpy(dtype=float)
    return {
        'Media aritmetică': valori.mean(),
        'Media geometrică': np.exp(np.log(valori).mean()),
        'Media armonică': len(valori) / np.sum(1 / valori)
    }

medii = pd.DataFrame({
    'Cantitate (unități)': indicatori_medii(date['transaction_qty']),
    'Preț unitar (USD)': indicatori_medii(date['unit_price'])
}).T
frecvente_cantitate = grupare_cantitate.iloc[:-1]['Frecvență'].to_numpy()
valori_cantitate = grupare_cantitate.iloc[:-1]['Cantitate cumpărată'].to_numpy(dtype=float)
centre_pret = (limite[:-1] + limite[1:]) / 2
frecvente_pret = grupare_pret.iloc[:-1]['Frecvență'].to_numpy()
medie_cantitate_grupata = np.average(valori_cantitate, weights=frecvente_cantitate)
medie_pret_grupata = np.average(centre_pret, weights=frecvente_pret)
comparatie_medii = pd.DataFrame({
    'Distribuție': ['Cantitate', 'Preț unitar'],
    'Medie exactă': [medii.loc['Cantitate (unități)', 'Media aritmetică'],
                    medii.loc['Preț unitar (USD)', 'Media aritmetică']],
    'Medie din grupare': [medie_cantitate_grupata, medie_pret_grupata]
})
comparatie_medii['Diferență (grupat − exact)'] = (
    comparatie_medii['Medie din grupare'] - comparatie_medii['Medie exactă']
)
assert np.isclose(medie_cantitate_grupata, date['transaction_qty'].mean())
display(medii.round(4))
display(comparatie_medii.round(4))

### 3.2. Indicatorii medii de poziție pentru fiecare distribuție

Mediana, quartilele și modulul exact provin din observații. La preț se prezintă și **clasa mediană** și **clasa modală** din tabelul Sturges; acestea indică un interval, nu o valoare exactă. Pentru cantitate, grupele sunt valori întregi individuale, astfel încât mediana și modulul grupării coincid cu cele exacte.

In [ ]:
def indicatori_pozitie(serie):
    frecvente = serie.value_counts()
    moduri = sorted(frecvente[frecvente == frecvente.max()].index.tolist())
    return {
        'Q1': serie.quantile(.25), 'Mediană': serie.median(),
        'Q3': serie.quantile(.75),
        'Modul': ', '.join(f'{valoare:g}' for valoare in moduri)
    }

pozitie = pd.DataFrame({
    'Cantitate (unități)': indicatori_pozitie(date['transaction_qty']),
    'Preț unitar (USD)': indicatori_pozitie(date['unit_price'])
}).T
frecvente_cumulate_pret = np.cumsum(frecvente_pret)
indice_clasa_mediana = np.searchsorted(frecvente_cumulate_pret, n / 2)
indice_clasa_modala = int(np.argmax(frecvente_pret))
clase_pret = pd.Series({
    'Clasa mediană a prețului': etichete_pret[indice_clasa_mediana],
    'Clasa modală a prețului': etichete_pret[indice_clasa_modala],
    'Mediană exactă a cantității': date['transaction_qty'].median(),
    'Modul exact al cantității': date['transaction_qty'].mode().iloc[0]
})
display(pozitie)
display(clase_pret)

### 3.3. Indicatorii simpli și sintetici ai variației

Amplitudinea este maximul minus minimul, iar amplitudinea interquartilică este `Q3 − Q1`. Dispersia și abaterea standard folosesc numitorul `n` (`ddof=0`), deoarece descriu toate înregistrările disponibile, nu estimează o variație a unei populații externe. Coeficientul de variație este `abaterea standard / media aritmetică × 100`; comparația sa între cantitate și preț trebuie interpretată prudent, deoarece variabilele au semnificații diferite, iar prețul combină categorii distincte. Aproximația pe intervale a variației prețului folosește centrele claselor.

In [ ]:
def indicatori_variatie(serie):
    return {
        'Minim': serie.min(), 'Maxim': serie.max(),
        'Amplitudine': serie.max() - serie.min(),
        'Amplitudine interquartilică': serie.quantile(.75) - serie.quantile(.25),
        'Dispersie (ddof=0)': serie.var(ddof=0),
        'Abatere standard (ddof=0)': serie.std(ddof=0),
        'Coeficient de variație (%)': serie.std(ddof=0) / serie.mean() * 100
    }

variatie = pd.DataFrame({
    'Cantitate (unități)': indicatori_variatie(date['transaction_qty']),
    'Preț unitar (USD)': indicatori_variatie(date['unit_price'])
}).T
dispersie_cantitate_grupata = np.average(
    (valori_cantitate - medie_cantitate_grupata) ** 2,
    weights=frecvente_cantitate
)
dispersie_pret_grupata = np.average(
    (centre_pret - medie_pret_grupata) ** 2, weights=frecvente_pret
)
comparatie_variatie = pd.DataFrame({
    'Distribuție': ['Cantitate', 'Preț unitar'],
    'Abatere standard exactă': [date['transaction_qty'].std(ddof=0),
                               date['unit_price'].std(ddof=0)],
    'Abatere standard din grupare': [np.sqrt(dispersie_cantitate_grupata),
                                    np.sqrt(dispersie_pret_grupata)]
})
assert np.isclose(dispersie_cantitate_grupata, date['transaction_qty'].var(ddof=0))
display(variatie.round(4))
display(comparatie_variatie.round(4))

### Concluzii generale ale cercetării statistice

Fișierul cuprinde 149.116 înregistrări pentru trei locații, fără identificatori repetați, valori lipsă sau zile lipsă în perioada studiată. Indicatorii exacți arată că o înregistrare conține în medie **1,4383 unități** (mediană: 1), iar prețul unitar mediu este **3,3822 USD** (mediană: 3,00 USD). Diferența dintre media și mediana prețului reflectă distribuția asimetrică și existența produselor cu preț ridicat din alte categorii. Aceste constatări descriu fișierul și nu certifică proveniența ori caracterul complet al vânzărilor reale.

# IV. Analiza și ajustarea seriei cronologice

### 4.1. Fenomenul studiat, sursa și perioada

Fenomenul este valoarea zilnică rezultată din `transaction_qty × unit_price`, însumată pentru cele trei locații. Seria are 181 de zile consecutive, 1 ianuarie – 30 iunie 2023. Este o **serie de flux**: fiecare nivel însumează valorile unei zile. Sursa și limitele sale sunt cele din 1.7. Sunt prezentate suplimentar numărul de înregistrări, unitățile și media pe zi pentru fiecare lună. Nu se impută sau elimină zile.

In [ ]:
serie_zilnica = (
    date_analiza.groupby('transaction_date')
    .agg(inregistrari=('transaction_id', 'size'),
         unitati=('transaction_qty', 'sum'),
         venit_usd=('venit_usd', 'sum'))
    .sort_index()
    .asfreq('D')
)
assert len(serie_zilnica) == len(zile_asteptate)
assert not serie_zilnica.isna().any().any()
assert serie_zilnica['inregistrari'].sum() == len(date_analiza)
assert serie_zilnica['unitati'].sum() == date_analiza['transaction_qty'].sum()
assert np.isclose(serie_zilnica['venit_usd'].sum(), date_analiza['venit_usd'].sum())

serie_lunara = serie_zilnica.resample('MS').agg({
    'inregistrari': 'sum', 'unitati': 'sum', 'venit_usd': 'sum'
})
serie_lunara['zile_observate'] = serie_zilnica['venit_usd'].resample('MS').size()
serie_lunara['inregistrari_pe_zi'] = (
    serie_lunara['inregistrari'] / serie_lunara['zile_observate']
)
serie_lunara['venit_pe_zi_usd'] = (
    serie_lunara['venit_usd'] / serie_lunara['zile_observate']
)
serie_lunara['venit_pe_inregistrare_usd'] = (
    serie_lunara['venit_usd'] / serie_lunara['inregistrari']
)
assert serie_lunara['inregistrari'].sum() == len(date)
assert np.isclose(serie_lunara['venit_usd'].sum(), serie_zilnica['venit_usd'].sum())
display(serie_lunara.round(2))

### 4.2. Indicatorii modificării absolute

Modificarea în lanț este `y_t − y_(t−1)`, iar modificarea față de bază este `y_t − y_1`, unde `y_t` este valoarea zilnică calculată. Prima zi nu are modificare în lanț. Tabelul complet rămâne în obiectul `serie_zilnica`; pentru lizibilitate sunt afișate primele și ultimele cinci zile, plus extrema modificărilor zilnice.

In [ ]:
serie_zilnica['modificare_lant_usd'] = serie_zilnica['venit_usd'].diff()
serie_zilnica['modificare_baza_usd'] = (
    serie_zilnica['venit_usd'] - serie_zilnica['venit_usd'].iloc[0]
)
rezumat_modificari = serie_zilnica['modificare_lant_usd'].agg(
    ['min', 'max', 'mean']
).rename({'min': 'Scădere zilnică maximă',
          'max': 'Creștere zilnică maximă',
          'mean': 'Modificare medie în lanț'})
display(pd.concat([serie_zilnica.head(5), serie_zilnica.tail(5)]).round(2))
display(rezumat_modificari.round(2))

### 4.3. Indicatorii modificării relative

Indicele în lanț este `y_t / y_(t−1) × 100`, iar indicele față de bază este `y_t / y_1 × 100`. Ritmul de creștere este indicele minus 100 puncte procentuale. Toate nivelurile zilnice sunt pozitive, deci rapoartele sunt definite. Valorile zilnice pot oscila mult; o singură zi nu caracterizează întreaga lună.

In [ ]:
serie_zilnica['indice_lant_pct'] = (
    serie_zilnica['venit_usd'] / serie_zilnica['venit_usd'].shift(1) * 100
)
serie_zilnica['indice_baza_pct'] = (
    serie_zilnica['venit_usd'] / serie_zilnica['venit_usd'].iloc[0] * 100
)
serie_zilnica['ritm_lant_pct'] = serie_zilnica['indice_lant_pct'] - 100
serie_zilnica['ritm_baza_pct'] = serie_zilnica['indice_baza_pct'] - 100
display(pd.concat([serie_zilnica.head(5), serie_zilnica.tail(5)])[ [
    'venit_usd', 'indice_lant_pct', 'ritm_lant_pct',
    'indice_baza_pct', 'ritm_baza_pct'
]].round(2))

### 4.4. Indicatorii medii ai seriei

Nivelul mediu zilnic este media aritmetică a celor 181 de niveluri. Modificarea absolută medie este `(y_n − y_1)/(n − 1)`, iar indicele mediu de dinamică este `(y_n / y_1)^(1/(n − 1)) × 100`. Ultimele două depind de zilele de început și sfârșit; de aceea se arată și venitul mediu pe zi în fiecare lună.

In [ ]:
niveluri = serie_zilnica['venit_usd']
indicatori_medii_serie = pd.Series({
    'Nivel mediu zilnic (USD)': niveluri.mean(),
    'Modificare absolută medie pe zi (USD)': (
        niveluri.iloc[-1] - niveluri.iloc[0]
    ) / (len(niveluri) - 1),
    'Indice mediu de dinamică (%)': (
        niveluri.iloc[-1] / niveluri.iloc[0]
    ) ** (1 / (len(niveluri) - 1)) * 100,
})
indicatori_medii_serie['Ritm mediu zilnic (%)'] = (
    indicatori_medii_serie['Indice mediu de dinamică (%)'] - 100
)
display(indicatori_medii_serie.round(4))
display(serie_lunara[['zile_observate', 'venit_pe_zi_usd',
                       'inregistrari_pe_zi',
                       'venit_pe_inregistrare_usd']].round(2))

### 4.5. Ajustarea seriei cronologice

Se estimează prin cele mai mici pătrate un trend liniar descriptiv `ŷ_t = a + b·t`, cu `t = 0,...,180`, pentru venitul zilnic. Se raportează `R²` și eroarea medie absolută (MAE) **pe aceleași zile folosite la ajustare**; acestea nu reprezintă verificare pe date viitoare. Media mobilă centrată pe șapte zile și media pe zi a săptămânii ajută la observarea fluctuațiilor. Nu se face extrapolare sau afirmație cauzală.

In [ ]:
t = np.arange(len(serie_zilnica), dtype=float)
y = serie_zilnica['venit_usd'].to_numpy(dtype=float)
panta, intercept = np.polyfit(t, y, deg=1)
serie_zilnica['trend_liniar_usd'] = intercept + panta * t
serie_zilnica['medie_mobila_7_zile_usd'] = (
    serie_zilnica['venit_usd'].rolling(7, center=True, min_periods=7).mean()
)
reziduuri = y - serie_zilnica['trend_liniar_usd'].to_numpy()
sse = np.sum(reziduuri ** 2)
sst = np.sum((y - y.mean()) ** 2)
diagnostic_trend = pd.Series({
    'Intercept (USD)': intercept,
    'Pantă (USD/zi)': panta,
    'R² în eșantion': 1 - sse / sst,
    'MAE în eșantion (USD)': np.mean(np.abs(reziduuri))
})
zile_saptamana = ['Luni', 'Marți', 'Miercuri', 'Joi',
                  'Vineri', 'Sâmbătă', 'Duminică']
medie_dupa_zi = (
    serie_zilnica.groupby(serie_zilnica.index.dayofweek)['venit_usd']
    .mean().rename(index=dict(enumerate(zile_saptamana)))
)
display(diagnostic_trend.round(3))
display(medie_dupa_zi.round(2).rename('Venit mediu zilnic (USD)').to_frame())

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(serie_zilnica.index, serie_zilnica['venit_usd'],
        alpha=.35, label='Venit zilnic')
ax.plot(serie_zilnica.index, serie_zilnica['medie_mobila_7_zile_usd'],
        linewidth=2, label='Medie mobilă centrată, 7 zile')
ax.plot(serie_zilnica.index, serie_zilnica['trend_liniar_usd'],
        linewidth=2, label='Trend liniar estimat')
ax.set(title='Venitul zilnic calculat și ajustarea în perioada observată',
       xlabel='Data', ylabel='Venit calculat (USD)')
ax.legend()
plt.tight_layout()
plt.show()

### Concluzii generale ale analizei seriei cronologice

Venitul calculat din fișier crește de la **81.677,74 USD în ianuarie** la **166.485,88 USD în iunie**. Cum lunile au durate diferite, comparația mai potrivită este media pe zi: **2.634,77 USD/zi** în ianuarie și **5.549,53 USD/zi** în iunie. Numărul mediu de înregistrări pe zi urcă de la **558,52** la **1.178,40**, în timp ce venitul calculat pe înregistrare rămâne aproape constant (**4,72** față de **4,71 USD**). În datele observate, creșterea valorii zilnice este asociată în principal cu mai multe înregistrări, nu cu o creștere clară a valorii medii pe înregistrare. Trendul liniar rezumă perioada studiată; fluctuațiile zilnice și lipsa datelor din afara intervalului nu permit atribuirea unei cauze sau prognoze sigure.